# Regresión PSBP sobre scores de FPCA (representación funcional)


# 1. Imports y rutas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import sys
from pathlib import Path
from datetime import datetime
import time
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

# Modulos de simulacion
from model_psbp_fd.pipelines import (
    FunctionalDomain,
    FARSimulator,
    gaussian_integral_kernel,
    build_integral_matrix,
)

# Estandarizador data 
from model_psbp_fd.functions_models import DataStandardizer

# Modulos de representacion funcional 
from model_psbp_fd.functions_models import FunctionalRepresentation

# Modulos de utilidades
from model_psbp_fd.utils import get_project_root

# Módulos de visualización 
SYS_PATH_GRAPHICS = Path("graphics")
if str(SYS_PATH_GRAPHICS) not in sys.path:
    sys.path.insert(0, str(SYS_PATH_GRAPHICS))

from model_psbp_fd.graphics import (
    # viz_traces
    plot_traces_bj, plot_traces_pj,
    plot_convergence_bj, plot_convergence_pj,
    # viz_global_components
    plot_global_components, plot_active_clusters,
    # viz_functional_data
    plot_empirical_sample, plot_functional_mean,
    plot_functional_variance, plot_mean_and_variance,
    # viz_time_series
    plot_fts_empirical, plot_fts_functional, plot_fts_comparison,
    # viz_prediction
    plot_scatter_theta, plot_functional_comparison,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## 1.1 Constantes a modificar según experimento 

In [ ]:
# Buscamos la raiz del proyecto
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT : {PROJECT_ROOT}")


# IDENTIFICAR EL EXPERIMENTO: variables globales y rutas de salida
BASENAME      = "modelo_unificado"
TT            = 1
SEED          = 4123
#TIMESTAMP     = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"{BASENAME}_{TT}"#_{TIMESTAMP}"      # Nombre del experimento 
print(f"Experiment ID : {EXPERIMENT_ID}")
print(f"Seed (base)   : {SEED}")

## 1.2 Construccion de Rutas

In [ ]:
# Construccion de rutas 
_REPORT_DIR = PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID
_ARTEFACT_DIR = PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,                         # Datos: raw + estandarizados
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,    # Coeficientes funcionales + FPCA
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,       # Datos predichos
    "out_report":   _REPORT_DIR,          # Figuras + métricas/config JSON
    "out_artefact": _ARTEFACT_DIR,        # Artefactos (serializados, por experimento)
    "out":          _REPORT_DIR,          # alias retrocompatible (= out_report): destino de PATHS["out"]
}
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:10s} → {path}")

# 2. Simulación y estandarización

In [ ]:
# CONSTANTES DE LA SIMULACION 
FAR_PARAMS = {
    "n_curves":       120,
    "n_points":       200,
    "bandwidth":      0.05,
    "decay":          0.8,
    "temporal_slope": 3.0,
    "spatial_slope":  2.0,
    "intercept":      1.0,
    "noise_std":      0.5,
    "noise_type":     "smooth",
    "burn_in":        100,
}
for k, v in FAR_PARAMS.items():
    print(f"  {k:<18}: {v}")

## 2.1 Simulación FAR

In [ ]:
domain = FunctionalDomain.regular(n_points=FAR_PARAMS["n_points"])
Psi    = build_integral_matrix(
    domain, gaussian_integral_kernel,
    bandwidth=FAR_PARAMS["bandwidth"],
    decay=FAR_PARAMS["decay"],
)

sim = FARSimulator(
    domain=domain,
    n_curves=FAR_PARAMS["n_curves"],
    Psi=Psi,
    trend="linear",
    trend_params={
        "temporal_slope": FAR_PARAMS["temporal_slope"],
        "spatial_slope":  FAR_PARAMS["spatial_slope"],
        "intercept":      FAR_PARAMS["intercept"],
    },
    noise_std=FAR_PARAMS["noise_std"],
    noise_type=FAR_PARAMS["noise_type"],
    burn_in=FAR_PARAMS["burn_in"],
    random_state=SEED,
)
X_raw = sim.simulate()   # (T, G)
T, G  = X_raw.shape
print(f"Datos simulados : {X_raw.shape}  →  T={T} curvas, G={G} puntos")

## 2.2 Visualización de los datos empíricos

In [ ]:
# ── Serie de tiempo funcional empírica (línea continua desplazada) ────────────
highlight_idx = [0, 1, sim.n_curves // 2, sim.n_curves - 1]

fig = plot_fts_empirical(
    X_raw, domain.grid,
    highlight_idx   = highlight_idx,
    title           = f"FAR(1) — {T} curvas empíricas (escala original)",
    separator_every = 5,
    save_path       = str(PATHS["out"] / "01_fts_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Muestra de curvas empíricas (5 índices fijos) ─────────────────────────────
fig = plot_empirical_sample(
    X_raw, domain.grid,
    sample_idx = [40, 1, 2, 80, 4],
    title      = "Muestra de 5 curvas empíricas (escala original)",
    save_path  = str(PATHS["out"] / "02_muestra_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Media y varianza funcional (panel combinado) ──────────────────────────────
fig = plot_mean_and_variance(
    X_raw, domain.grid,
    show_std1 = True,
    show_std2 = True,
    title     = "Media y varianza funcional — FAR(1) escala original",
    save_path = str(PATHS["out"] / "03_media_varianza_raw.png"),
)
plt.show()

## 2.3 Estandarización por columna 

In [ ]:
# CONSTANTES DE ESTANDARIZACION O TRANSFORMACION DE LOS DATOS
standardizer = DataStandardizer(method="zscore_column", ddof=0)
X = standardizer.fit_transform(X_raw)

print(standardizer.summary())
print(f"X estandarizada : shape={X.shape}")
print(f"  max|media|  = {np.abs(X.mean(axis=0)).max():.2e}  (debe ≈ 0)")
print(f"  max|std-1|  = {np.abs(X.std(axis=0) - 1).max():.2e}  (debe ≈ 0)")

# ── Guardar artefactos de estandarización ───────────────────────────────
standardizer.save(PATHS["out_artefact"])

# ── Guardar datos ───────────────────────────────────────────────────────
np.savetxt(PATHS["raw"] / "datos_raw.csv", X_raw, delimiter=",")
np.savetxt(PATHS["raw"] / "datos_transform.csv", X, delimiter=",")

print(f"[raw] datos crudos {X_raw.shape} · datos transformados {X.shape}")
print(f"      → {PATHS['raw']}")

In [ ]:
# ── Diagnóstico visual de estandarización ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, Xp, lbl in zip(axes, [X_raw, X], ["Original", "Estandarizada"]):
    ax.plot(domain.grid, Xp.mean(axis=0), color="steelblue", lw=1.5, label="media")
    ax.plot(domain.grid, Xp.std(axis=0),  color="crimson",   lw=1.5, ls="--", label="std")
    ax.axhline(0, color="k", lw=0.5)
    ax.axhline(1, color="k", lw=0.5, ls=":")
    ax.set_title(f"X {lbl}: estadísticas marginales por columna")
    ax.set_xlabel("s"); ax.legend(fontsize=8); ax.grid(True, alpha=0.4)

plt.tight_layout()
fig.savefig(PATHS["out"] / "04_diagnostico_estandarizacion.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# ── Series de tiempo funcionales: original vs estandarizada ──────────────────
fig = plot_fts_empirical(
    X_raw, domain.grid,
    highlight_idx=highlight_idx,
    title=f"FAR(1) — {T} curvas (escala ORIGINAL)",
    separator_every=5,
    save_path=str(PATHS["out"] / "05_fts_empirica_original.png"),
)
plt.show()

fig = plot_fts_empirical(
    X, domain.grid,
    highlight_idx=highlight_idx,
    color="#3aaa35",
    title=f"FAR(1) — {T} curvas (ESTANDARIZADA por columna)",
    separator_every=5,
    save_path=str(PATHS["out"] / "06_fts_empirica_std.png"),
)
plt.show()

In [ ]:
# ── Media y varianza funcional de X estandarizada ────────────────────────────
fig = plot_mean_and_variance(
    X, domain.grid,
    show_std1 = True,
    title     = "Media y varianza funcional — FAR(1) estandarizada",
    save_path = str(PATHS["out"] / "07_media_varianza_std.png"),
)
plt.show()

# 3. Representación funcional B-spline

In [ ]:
# SELECION DE PARAMETROS PARA EVALUAR LA REPRESENTACION FUNCIONAL (n_basis, order)
N_BASIS_RANGE = range(2, min(30, T // 2))
ORDER_RANGE   = range(1, 5)
selection_records = []

for order in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < order:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=order)
            TH_tmp = fr_tmp.fit_transform(X, domain.grid)
            X_rec  = fr_tmp.reconstruct(TH_tmp)
            ss_res = np.sum((X - X_rec) ** 2)
            ss_tot = np.sum((X - X.mean(axis=0, keepdims=True)) ** 2)
            vr     = 1.0 - ss_res / ss_tot
            rmse_c = np.sqrt(np.mean((X - X_rec) ** 2, axis=1))
            selection_records.append({
                "n_basis": nb, "order": order, "var_retained": vr,
                "rmse_mean": rmse_c.mean(), "rmse_max": rmse_c.max(),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={order}: {e}")

sel_df   = pd.DataFrame(selection_records)
best_idx = sel_df.sort_values(["var_retained", "n_basis"], ascending=[False, True]).index[0]
best_row = sel_df.loc[best_idx]
nb_best  = int(best_row["n_basis"])
ord_best = int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}", "rmse_max": "{:.6f}"})
    .background_gradient(subset=["var_retained"], cmap="YlGn")
    .background_gradient(subset=["rmse_mean"],    cmap="YlOrRd_r"))
print(f"\nRecomendación: n_basis={nb_best}, order={ord_best}, var_retained={best_row['var_retained']:.4%}")

In [ ]:
# ── Heatmaps + curva de varianza retenida ────────────────────────────────────
vr_pivot   = sel_df.pivot(index="order", columns="n_basis", values="var_retained")
rmse_pivot = sel_df.pivot(index="order", columns="n_basis", values="rmse_mean")
best_nb_idx  = list(vr_pivot.columns).index(nb_best)
best_ord_idx = list(vr_pivot.index).index(ord_best)
orders_list  = sorted(sel_df["order"].unique())
colors_ord   = ["#1a6faf", "#e07b39", "#3aaa35", "#9b59b6"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, pivot, cmap, label in [
    (axes[0], vr_pivot*100,   "YlGn",    "Varianza retenida (%)"),
    (axes[1], rmse_pivot,     "YlOrRd_r", "RMSE medio de reconstrucción"),
]:
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, origin="lower")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index,   fontsize=8)
    ax.set_xlabel("n_basis"); ax.set_ylabel("order"); ax.set_title(label)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.3g}", ha="center", va="center", fontsize=6.5)
    ax.add_patch(plt.Rectangle(
        (best_nb_idx-.5, best_ord_idx-.5), 1, 1,
        fill=False, edgecolor="crimson", lw=2.5, label="recomendado"))
    ax.legend(fontsize=7, loc="upper left")
    plt.colorbar(im, ax=ax, shrink=0.8)

for i, ord_ in enumerate(orders_list):
    sub = sel_df[sel_df["order"] == ord_].sort_values("n_basis")
    axes[2].plot(sub["n_basis"], sub["var_retained"]*100, marker="o", lw=1.4, ms=5,
                 color=colors_ord[i % 4], label=f"order={ord_}")
axes[2].axvline(nb_best, color="crimson", ls="--", lw=1.2, label=f"nb_best={nb_best}")
axes[2].set_xlabel("n_basis"); axes[2].set_ylabel("Varianza retenida (%)")
axes[2].set_title("Varianza retenida vs n_basis"); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
fig.savefig(PATHS["out"] / "08_seleccion_basis.png", dpi=130, bbox_inches="tight")
plt.show()

## 3.1 Ajuste y visualización de la representación funcional

In [ ]:
#CONSTANTES ELEGIDAS PARA LA REPRESENTACION FUNCIONAL
fr    = FunctionalRepresentation(method="bspline", n_basis=29, order=3)

# Ajuste
THETA = fr.fit_transform(X, domain.grid)
print(f"THETA shape: {THETA.shape}  (T={THETA.shape[0]}, K={THETA.shape[1]})")

# ── Serie de tiempo funcional con representación B-spline ────────────────────
fig = plot_fts_functional(
    X, domain.grid,
    fr              = fr,
    highlight_idx   = highlight_idx,
    title           = f"FAR(1) — {T} curvas (repr. B-spline, n_basis={nb_best}, order={ord_best})",
    separator_every = 5,
    save_path       = str(PATHS["out"] / "09_fts_funcional_bspline.png"),
)
plt.show()

## 3.2 Pasar la base selecionada a FPCA analizar cuantos componentes me quedare

In [ ]:
# Base B-spline NO ortonormal ⇒ Gram W = <φ_j,φ_k>_{L²} ≠ I. La FPCA correcta
# diagonaliza en métrica L²:  (W^{1/2} S_θ W^{1/2}) u = λ u,  ‖ψ_m‖_{L²}=1.
# ----------------------------------------------------------------------------
K = THETA.shape[1]

# ── Base en grilla + verificación de linealidad de reconstruct ───────────────
Phi = fr.reconstruct(np.eye(K)).T                        # (G, K): columna k = φ_k en grilla
_lin_err = np.abs(fr.reconstruct(THETA) - THETA @ Phi.T).max()
assert _lin_err < 1e-8, (
    f"reconstruct no es el mapa lineal Θ·Φᵀ esperado (err={_lin_err:.2e})."
)

# ── Gram L² (cuadratura trapezoidal) ─────────────────────────────────────────
gpts = np.asarray(domain.grid, dtype=float)
w_quad = np.empty_like(gpts)
w_quad[1:-1] = (gpts[2:] - gpts[:-2]) / 2.0
w_quad[0]    = (gpts[1]  - gpts[0])  / 2.0
w_quad[-1]   = (gpts[-1] - gpts[-2]) / 2.0
W = Phi.T @ (w_quad[:, None] * Phi); W = 0.5 * (W + W.T)

# ── Covarianza de coeficientes y problema propio simetrizado en L² ───────────
mu_theta = THETA.mean(axis=0)
Theta_c  = THETA - mu_theta
S_theta  = (Theta_c.T @ Theta_c) / (THETA.shape[0] - 1)
evW, VW    = np.linalg.eigh(W); evW = np.clip(evW, 1e-12, None)
W_half     = VW @ np.diag(np.sqrt(evW))   @ VW.T
W_half_inv = VW @ np.diag(1 / np.sqrt(evW)) @ VW.T
M_sym = W_half @ S_theta @ W_half; M_sym = 0.5 * (M_sym + M_sym.T)
evals, U = np.linalg.eigh(M_sym)
order = np.argsort(evals)[::-1]
evals = np.clip(evals[order], 0.0, None); U = U[:, order]
B_full = W_half_inv @ U                                  # (K, K) coef. autofunciones (b_mᵀ W b_m = 1)
var_ratio = evals / evals.sum(); var_cum = np.cumsum(var_ratio)

# ── Diagnóstico DINÁMICO: AR(1) propio de cada componente (≠ varianza) ───────
# Varianza = representación; AR(1) = dinámica. Una FPC de var. baja puede tener
# AR fuerte (útil para pronóstico) y una de var. alta puede ser ruido temporal.
SCORES_full = Theta_c @ (W @ B_full)                     # (T, K) scores de TODAS las componentes
s0, s1 = SCORES_full[:-1], SCORES_full[1:]
ar1_own = (s1 * s0).sum(0) / np.clip((s0 * s0).sum(0), 1e-12, None)   # pendiente AR(1) por comp.

# ── Sugerencia automática (SOLO referencia; no vincula nada) ─────────────────
VAR_TARGET  = 0.99
M_SUGGERIDO = int(np.clip(np.searchsorted(var_cum, VAR_TARGET) + 1, 1, K))

# ── Tabla ─────────────────────────────────────────────────────────────────────
fpca_tbl = pd.DataFrame({
    "componente": np.arange(1, K + 1),
    "autovalor":  evals,
    "var_ratio":  var_ratio,
    "var_acum":   var_cum,
    "ar1_propio": ar1_own,
})
display(fpca_tbl.head(min(15, K)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}", "var_acum": "{:.4%}", "ar1_propio": "{:+.3f}"}
).background_gradient(subset=["var_ratio"], cmap="YlGn")
 .background_gradient(subset=["ar1_propio"], cmap="coolwarm", vmin=-1, vmax=1))
print(f"\nK B-spline disponibles : {K}")
print(f"SUGERENCIA (var ≥ {VAR_TARGET:.0%}) : M_SUGGERIDO = {M_SUGGERIDO}   "
      f"← solo referencia; fija M_FPCA en la celda 3.4")

# ── Scree + varianza acumulada + AR(1) por componente ────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(np.arange(1, K + 1), evals, "o-")
ax[0].axvline(M_SUGGERIDO, ls="--", c="crimson", label=f"sug. M={M_SUGGERIDO}")
ax[0].set(title="Scree (λ_m)", xlabel="componente", ylabel="λ_m"); ax[0].set_yscale("log"); ax[0].legend()
ax[1].plot(np.arange(1, K + 1), var_cum, "o-")
ax[1].axhline(VAR_TARGET, ls=":", c="grey"); ax[1].axvline(M_SUGGERIDO, ls="--", c="crimson")
ax[1].set(title="Varianza acumulada", xlabel="componente", ylabel="proporción", ylim=(0, 1.02))
fig.tight_layout()
fig.savefig(PATHS["out"] / "10_fpca_scree.png", dpi=110, bbox_inches="tight")
plt.show()

## 3.3 Obtener el FPCA basado en la selecion

In [ ]:
M_FPCA = 9    # ← nº de componentes FPCA a retener (lo fijas TÚ)

# ── Validación (no avanza hasta que elijas) ──────────────────────────────────
assert M_FPCA is not None, "Define M_FPCA (entero) en esta celda antes de continuar."
assert isinstance(M_FPCA, (int, np.integer)) and 1 <= M_FPCA <= K, (
    f"M_FPCA debe ser entero en [1, {K}]. Recibido: {M_FPCA!r}"
)
M_fpca = int(M_FPCA)                                     # alias en minúscula para el resto del notebook

# ── Construcción con el M elegido ────────────────────────────────────────────
B        = B_full[:, :M_fpca]                            # (K, M)
Psi_grid = Phi @ B                                       # (G, M) autofunciones (‖ψ_m‖_{L²}=1)
mu_grid  = Phi @ mu_theta                                # (G,)
SCORES   = Theta_c @ (W @ B)                             # (T, M) scores ξ

# ── Sanidad de construcción: autofunciones ortonormales en L² (Gram ≈ I) ─────
gram_psi = Psi_grid.T @ (w_quad[:, None] * Psi_grid)
print(f"[sanidad] max|<ψ_i,ψ_j> - I| : {np.abs(gram_psi - np.eye(M_fpca)).max():.2e}   (≈ 0)")

# EVALUACIÓN DE CORRELACIÓN  (lag 0)
Gamma0 = np.atleast_2d(np.cov(SCORES, rowvar=False, ddof=1))     # covarianza lag 0
R0     = np.atleast_2d(np.corrcoef(SCORES, rowvar=False))        # correlación lag 0
off_R0 = np.abs(R0 - np.eye(M_fpca))

print(f"max|media ξ|                 : {np.abs(SCORES.mean(0)).max():.2e}   (≈ 0)")
print(f"max|diag(Γ0) - λ|            : {np.abs(np.diag(Gamma0) - evals[:M_fpca]).max():.2e}   (≈ 0)")
print(f"max|correlación fuera diag.| : {off_R0.max():.2e}   (≈ 0 ⇒ sin correlación contemporánea)")

if M_fpca >= 2:
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(R0, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(r"$R_0$: correlación contemporánea (lag 0)")
    ax.set_xlabel("FPC"); ax.set_ylabel("FPC")
    ax.set_xticks(range(M_fpca)); ax.set_yticks(range(M_fpca))
    ax.set_xticklabels(range(1, M_fpca + 1)); ax.set_yticklabels(range(1, M_fpca + 1))
    plt.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(PATHS["out"] / "11_fpca_correlacion_lag0.png", dpi=110, bbox_inches="tight")
    plt.show()

# ── Empaquetado FPCA + operador de reconstrucción (inversa de FPCA) ──────────
def reconstruct_from_scores(S_hat):
    """X̂(t) = μ(t) + Σ_m ξ_m ψ_m(t)."""
    S_hat = np.atleast_2d(np.asarray(S_hat, dtype=float))
    return mu_grid[None, :] + S_hat @ Psi_grid.T

FPCA = {
    "M": M_fpca, "eigvals": evals[:M_fpca], "mean_grid": mu_grid,
    "eigfun_grid": Psi_grid, "coef_eigfun": B, "gram": W,
    "reconstruct": reconstruct_from_scores,
}

# ── Persistencia de datos · representación funcional + FPCA → data/.../functional ──
np.savetxt(PATHS["functional"] / "theta.csv",               THETA,    delimiter=",")
np.savetxt(PATHS["functional"] / "basis_phi.csv",           Phi,      delimiter=",")
np.savetxt(PATHS["functional"] / "grid.csv",                gpts,     delimiter=",")
np.savetxt(PATHS["functional"] / "fpca_scores.csv",         SCORES,   delimiter=",")
np.savetxt(PATHS["functional"] / "fpca_eigenfunctions.csv", Psi_grid, delimiter=",")
np.savetxt(PATHS["functional"] / "fpca_mean.csv",           mu_grid,  delimiter=",")
with open(PATHS["functional"] / "fpca_meta.json", "w", encoding="utf-8") as _f:
    json.dump({"M": int(M_fpca), "K": int(THETA.shape[1]),
               "eigvals": FPCA["eigvals"].tolist(),
               "var_explained": float(var_cum[M_fpca - 1]), "var_target": VAR_TARGET},
              _f, indent=2, ensure_ascii=False)

print(f"\nSCORES shape : {SCORES.shape}   (T={SCORES.shape[0]}, M={M_fpca})")

# 4. Construcción de datasets AR(p) sobre scores de FPCA (ecuación-por-ecuación)

## 4.1 Analisis de rezasgos

In [ ]:
T_theta = SCORES.shape[0]
K_total = SCORES.shape[1]                                # nº de FPC disponibles (M)

# Lags de Diagnostico
N_LAGS_MAX = 3
N_LAGS_MAX = int(np.clip(N_LAGS_MAX, 1, T_theta - 2))

def _spearman_block(Y, X):
    """Spearman columna-a-columna vía rangos (pandas; sin scipy)."""
    Yr = pd.DataFrame(Y).rank().to_numpy()
    Xr = pd.DataFrame(X).rank().to_numpy()
    Yc = Yr - Yr.mean(0); Xc = Xr - Xr.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

# ── Matrices de correlación: respuesta (t) × predictor (FPC j, lag ℓ) ────────
n_cov = K_total * N_LAGS_MAX
corr_pearson  = np.zeros((K_total, n_cov))
corr_spearman = np.zeros((K_total, n_cov))
col_labels = []
y_block = SCORES[N_LAGS_MAX:, :]                         # respuesta en t (alineada)

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES[N_LAGS_MAX - lag : T_theta - lag, :]   # predictores en t-lag
    sp = _spearman_block(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_pearson[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_spearman[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band = 1.96 / np.sqrt(len(y_block))                      # banda de significancia (referencia)

# ── Estilo por |r| ────────────────────────────────────────────────────────────
def _text_style(val):
    a = abs(val)
    if   a >= 0.7: return 10, "bold",   "normal", "gold"      # alta
    elif a >= 0.5: return 9,  "bold",   "normal", "#cccccc"   # mod-alta
    elif a >= 0.3: return 9,  "normal", "italic", None        # moderada
    else:          return 8,  "normal", "normal", None        # baja

VCLIP = 0.6
norm  = mcolors.Normalize(vmin=-VCLIP, vmax=VCLIP, clip=True)

def _draw_heatmap(matrix, title, fname):
    fig, ax = plt.subplots(figsize=(0.9 * n_cov + 2, 1.4 * K_total + 1.2))
    im = ax.imshow(matrix, cmap="RdBu_r", norm=norm, aspect="auto")
    ax.set_xticks(range(n_cov)); ax.set_xticklabels(col_labels, fontsize=10)
    ax.set_yticks(range(K_total)); ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_xlabel("predictor (FPC, lag)"); ax.set_ylabel("respuesta (t)")
    ax.set_title(title, fontsize=11)
    for lag in range(1, N_LAGS_MAX):                     # separadores entre bloques de lag
        ax.axvline(lag * K_total - 0.5, color="black", lw=1.2)
    for i in range(K_total):
        for j in range(n_cov):
            val = matrix[i, j]; fs, fw, fst, box = _text_style(val)
            bbox = (dict(boxstyle="round,pad=0.15", facecolor=box,
                         edgecolor="dimgray", alpha=0.85, lw=0.8) if box else None)
            ax.text(j, i, f"{val:+.2f}", ha="center", va="center",
                    color=("white" if abs(val) > 0.35 else "black"),
                    fontsize=fs, fontweight=fw, fontstyle=fst, bbox=bbox)
    legend_elements = [
        mpatches.Patch(facecolor="gold",    edgecolor="dimgray", label="|r| ≥ 0.7 (alta)"),
        mpatches.Patch(facecolor="#cccccc", edgecolor="dimgray", label="|r| ≥ 0.5 (mod-alta)"),
        mpatches.Patch(facecolor="white",   edgecolor="white",   label="|r| ≥ 0.3 (mod)"),
        mpatches.Patch(facecolor="white",   edgecolor="white",   label=f"banda ±{band:.2f}"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=7.5,
              framealpha=0.9, title="Nivel", title_fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85, label=f"correlación (sat. ±{VCLIP})")
    fig.tight_layout()
    fig.savefig(PATHS["out"] / fname, dpi=110, bbox_inches="tight")
    plt.show()

_draw_heatmap(corr_pearson,  f"Pearson — respuesta(t) vs lags 1..{N_LAGS_MAX}",  "12a_rezagos_pearson.png")
_draw_heatmap(corr_spearman, f"Spearman — respuesta(t) vs lags 1..{N_LAGS_MAX}", "12b_rezagos_spearman.png")


## 4.2 Selecion de lags y DATSET final 

In [ ]:
N_LAGS  = 2
T_theta = SCORES.shape[0]
K_total = SCORES.shape[1]   # nº de FPC disponibles (M)
T_eff   = T_theta - N_LAGS
COMPONENT_IDX = list(range(K_total)) 

print(f"K_total disponibles : {K_total}  (índices 0 … {K_total - 1})")
print(f"T_theta             : {T_theta}")
print(f"N_LAGS              : {N_LAGS}")
print(f"T_eff               : {T_eff}")

In [ ]:
# ── Validación ───────────────────────────────────────────────────────────────
assert len(COMPONENT_IDX) > 0, "COMPONENT_IDX no puede estar vacío."
assert len(COMPONENT_IDX) == len(set(COMPONENT_IDX)), "COMPONENT_IDX tiene índices repetidos."
assert all(0 <= i < K_total for i in COMPONENT_IDX), (
    f"Todos los índices deben estar en [0, {K_total - 1}]. Recibido: {COMPONENT_IDX}"
)

n_components = len(COMPONENT_IDX)

# ── Resumen ───────────────────────────────────────────────────────────────────
print(f"K_total disponibles : {K_total}")
print(f"Componentes usados  : {n_components}  →  índices {COMPONENT_IDX}")
print(f"Orden AR (N_LAGS)   : {N_LAGS}")
print()
print(f"  {'k_modelo':>8}  {'idx_THETA':>10}  {'nombre_resp':>14}")
print(f"  {'─'*8}  {'─'*10}  {'─'*14}")
for k_model, idx in enumerate(COMPONENT_IDX):
    print(f"  {k_model:>8}  {idx:>10}  {'fpc_' + str(idx + 1):>14}")

In [ ]:
# ── Construcción de DataFrames AR(p) ─────────────────────────────────────────
SCORES_sel = SCORES[:, COMPONENT_IDX]   # (T, n_components) — scores de FPCA

cov_names = [
    f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
    for lag in range(1, N_LAGS + 1)
    for j in range(n_components)
]

dfs = {}
for k in range(n_components):
    y_col  = SCORES_sel[N_LAGS:, k]
    X_cols = np.hstack([
        SCORES_sel[N_LAGS - lag: T_theta - lag, :]
        for lag in range(1, N_LAGS + 1)
    ])
    resp_name = f"fpc_{COMPONENT_IDX[k] + 1}"
    dfs[k] = pd.DataFrame(
        np.column_stack([y_col, X_cols]),
        columns=[resp_name] + cov_names,
    )

manifest = {
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "datasets":      {},
}
for k in range(n_components):
    fname = f"dataset_fpc_{COMPONENT_IDX[k] + 1}.csv"
    dfs[k].to_csv(PATHS["functional"] / fname, index=False)
    manifest["datasets"][fname] = {
        "response": f"fpc_{COMPONENT_IDX[k] + 1}",
        "shape":    list(dfs[k].shape),
    }
    print(f"  guardado {fname}  resp='fpc_{COMPONENT_IDX[k] + 1}'  shape={dfs[k].shape}")

with open(PATHS["functional"] / "datasets_manifest.json", "w", encoding="utf-8") as _f:
    json.dump(manifest, _f, indent=2, ensure_ascii=False)
print(f"[functional] {n_components} datasets + datasets_manifest.json → {PATHS['functional']}")

# 5. Especificación de hiperparámetros y ajuste MCMC

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 15, "M": 50}
N_CHAINS    = 2
CHAIN_SEEDS = [SEED + c * 100 for c in range(N_CHAINS)]
BURN        = int(MCMC_CONFIG["burn"])
print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS}  |  CHAIN_SEEDS: {CHAIN_SEEDS}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — priors heterogéneas por tipo de variable
# ════════════════════════════════════════════════════════════════════════════

# ── Escalares globales ────────────────────────────────────────────────────────
HP_GLOBAL = {
    "atau":  2.0,
    "btau":  0.5,
    "ag":    2.0,
    "bg":    0.5,
    "mumu":  0.0,
    "taumu": 1.0,
    "pwj":   0.5,
}

# ── Priors por tipo de variable ───────────────────────────────────────────────
#   Formato: "tipo": (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0,  0.0, 1.0),   # E[π] = 0.90
    "cross_lag": (1.0, 1.0,  0.0, 1.0),   # E[π] = 0.50
}

# ── Clasificador ──────────────────────────────────────────────────────────────
def _classify(name: str, k_model: int, component_idx: list) -> str:
    own_name = f"fpc_{component_idx[k_model] + 1}_lag1"
    if name == own_name:
        return "own_lag1"
    if "_lag" in name:
        return "cross_lag"
    return "cross_lag"   # fallback: cualquier variable no reconocida → débil

# ── Construcción automática de HYPERPARAMS_LIST ───────────────────────────────
HYPERPARAMS_LIST = []
for k in range(n_components):
    p = len(cov_names)
    apij    = np.empty(p); bpij    = np.empty(p)
    mupsij  = np.empty(p); taupsij = np.empty(p)

    for j, name in enumerate(cov_names):
        vtype              = _classify(name, k, COMPONENT_IDX)
        a, b, mu, tau      = HP_BY_TYPE[vtype]
        apij[j]    = a;    bpij[j]    = b
        mupsij[j]  = mu;   taupsij[j] = tau

    HYPERPARAMS_LIST.append({**HP_GLOBAL,
                              "apij": apij,     "bpij": bpij,
                              "mupsij": mupsij, "taupsij": taupsij})

# ── Tabla de resumen ──────────────────────────────────────────────────────────
_W = 70
print("═" * _W)
print(f"  HYPERPARAMS_LIST  —  {n_components} componentes × {p} variables")
print("═" * _W)
print(f"  Globales: atau={HP_GLOBAL['atau']} btau={HP_GLOBAL['btau']}  "
      f"ag={HP_GLOBAL['ag']} bg={HP_GLOBAL['bg']}  "
      f"mumu={HP_GLOBAL['mumu']} taumu={HP_GLOBAL['taumu']}  "
      f"pwj={HP_GLOBAL['pwj']}")
print()
for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"  Componente k={k+1}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'Variable':<26} {'Tipo':<12} {'apij':>6} {'bpij':>6} "
          f"{'E[π]':>6} {'mupsij':>8} {'taupsij':>9}")
    print(f"  {'─'*26} {'─'*12} {'─'*6} {'─'*6} "
          f"{'─'*6} {'─'*8} {'─'*9}")
    for j, name in enumerate(cov_names):
        vtype = _classify(name, k, COMPONENT_IDX)
        a  = hp["apij"][j];    b  = hp["bpij"][j]
        mu = hp["mupsij"][j];  tau = hp["taupsij"][j]
        e_pi = a / (a + b)
        marker = "  ◄" if vtype == "own_lag1" else ""
        print(f"  {name:<26} {vtype:<12} {a:>6.1f} {b:>6.1f} "
              f"{e_pi:>6.3f} {mu:>8.1f} {tau:>9.1f}{marker}")
    print()